# Red Team Evaluation - Playwright (Browser UI) Target

Evaluate a web-based chat interface using browser automation. Each attack session runs in its own browser tab so conversations stay isolated.

Update the CSS selectors and URL to match your chat UI.

**Prerequisites:**
- `pip install -r ../../requirements.txt`
- `pip install playwright && playwright install chromium`
- `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` in `../../.env`

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import asyncio
import os

from dotenv import load_dotenv
from hiddenlayer import AsyncHiddenLayer
from playwright.async_api import async_playwright

load_dotenv("../../.env")

hl_client = AsyncHiddenLayer(
    client_id=os.getenv("HIDDENLAYER_CLIENT_ID"),
    client_secret=os.getenv("HIDDENLAYER_CLIENT_SECRET"),
)

## Configurables

1. Configure browser
2. Configure target
3. Configure evaluation

In [ ]:
# Browser configuration
CHAT_URL = "https://chat.example.com"
NEW_SESSION_BUTTON_SELECTOR = "button.new-chat"
INPUT_SELECTOR = "textarea#chat-input"
OUTPUT_SELECTOR = "div.message.assistant"
MESSAGE_CONTENT_SELECTOR = ".message-content"
SUBMIT_METHOD = "enter"  # Options: "enter", "button"
SUBMIT_BUTTON_SELECTOR = "#send-btn"
HEADLESS = False

# Target configuration
TARGET_MODEL_NAME = "browser-ui-model"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."

# Evaluation configuration
EVAL_NAME = "Browser UI Red Team Eval"
MAX_CONCURRENT_TABS = 10
PARALLEL_TECHNIQUES = MAX_CONCURRENT_TABS
SESSIONS_PER_TECHNIQUE = 1
MAX_TURNS = 5
HIDDENLAYER_PROJECT_ID = None  # Options: None, <project_id>

## Initialize Browser and Create Handler

The handler drives one browser tab per attack session: it types the attack prompt, submits it, waits for a new assistant message, and returns its text.

In [ ]:
browser_context = {"playwright": None, "browser": None}
browser_context["playwright"] = await async_playwright().start()
browser_context["browser"] = await browser_context["playwright"].chromium.launch(headless=HEADLESS)

session_pages = {}
session_message_counts = {}


async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and target (one tab per session)."""
    if session_id not in session_pages:
        page = await browser_context["browser"].new_page()
        await page.goto(CHAT_URL)

        if NEW_SESSION_BUTTON_SELECTOR:
            await page.wait_for_selector(NEW_SESSION_BUTTON_SELECTOR, timeout=10000)
            await page.click(NEW_SESSION_BUTTON_SELECTOR)
            await asyncio.sleep(0.5)

        await page.wait_for_selector(INPUT_SELECTOR, timeout=10000)
        session_pages[session_id] = page
        session_message_counts[session_id] = 0

    page = session_pages[session_id]
    initial_count = session_message_counts[session_id]

    await page.fill(INPUT_SELECTOR, prompt)
    if SUBMIT_METHOD == "enter":
        await page.press(INPUT_SELECTOR, "Enter")
    else:
        await page.click(SUBMIT_BUTTON_SELECTOR)

    expected_count = initial_count + 1
    while True:
        messages = await page.query_selector_all(OUTPUT_SELECTOR)
        if len(messages) >= expected_count:
            last_message = messages[-1]
            if MESSAGE_CONTENT_SELECTOR:
                content_el = await last_message.query_selector(MESSAGE_CONTENT_SELECTOR)
                response_text = await (content_el or last_message).inner_text()
            else:
                response_text = await last_message.inner_text()

            if response_text:
                session_message_counts[session_id] = len(messages)
                return response_text

        await asyncio.sleep(1)

## Run the Evaluation

Open a red team session and run attack techniques in parallel across browser tabs.

In [ ]:
session = await hl_client.evaluation_sessions.red_team.start_session(
    name=EVAL_NAME,
    target_model=TARGET_MODEL_NAME,
    target_system_prompt=TARGET_SYSTEM_PROMPT,
    max_parallel_techniques=PARALLEL_TECHNIQUES,
    sessions_per_technique=SESSIONS_PER_TECHNIQUE,
    max_turns=MAX_TURNS,
    hiddenlayer_project_id=HIDDENLAYER_PROJECT_ID,
)

print(f"Session started: {session.workflow_id}")

try:
    await session.run_with_callback_parallel(handler=handler)
    print("Evaluation complete. View transcripts at https://console.hiddenlayer.ai")
finally:
    for page in session_pages.values():
        await page.close()
    session_pages.clear()
    session_message_counts.clear()
    await browser_context["browser"].close()
    await browser_context["playwright"].stop()

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await hl_client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")